# Fine-tuning a small model on the Kestrel Home support VOICE

**What this teaches:** how to fine-tune a small LLM to *reply in the Kestrel Home support house format/voice* using **QLoRA with Unsloth**, on a **free Google Colab T4 GPU** in about 10 minutes.

We are teaching the model a **format/voice**, not facts. Every Kestrel support reply follows the same five-part shape:

1. a short greeting line
2. `Answer:` a direct answer
3. `Policy:` the relevant policy / citation
4. `Next step:` what to do or the escalation path
5. the sign-off `— Kestrel Home Support`

**The before/after promise:**
- *Before* fine-tuning, the base model answers Kestrel questions in a generic, free-form, formatless way.
- *After* fine-tuning on ~120 in-house examples, the model reliably produces the exact Kestrel five-part format — the greeting, the `Answer:` / `Policy:` / `Next step:` markers, and the sign-off.

> Later in the notebook we prove the flip side: the fine-tuned model still gets **new facts wrong**, because fine-tuning taught the *voice*, not the *facts*. For facts you use **RAG** (last session). Production = **fine-tune for voice + RAG for facts**.


## 1. Install Unsloth

Colab already ships with a compatible PyTorch + CUDA, so we only need to install Unsloth (it pulls in `transformers`, `trl`, `peft`, `bitsandbytes`, and `datasets`).

> **Runtime check:** make sure you are on a GPU runtime — `Runtime → Change runtime type → T4 GPU`. The free T4 is plenty for a 1B model with QLoRA.


In [ ]:
# Colab already has a working torch/CUDA. Just add Unsloth.
# The plain install lets Unsloth pull a compatible trl/transformers/peft set.
!pip install unsloth

# --- FALLBACK: only if you hit a checkpoint-save PicklingError on trainer.train() ---
#   PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>:
#   it's not the same object as trl.trainer.sft_config.SFTConfig
# That means the installed trl/unsloth pair ended up with two copies of the
# SFTConfig class. Section 5 already guards against it (it passes an SFTConfig,
# not a TrainingArguments) — but if it still fires, pin a known-good pair, then
# Runtime -> Restart runtime -> Run all. Version numbers are illustrative; bump
# to the latest compatible set if these have aged out.
# !pip install -U "unsloth==2025.1.1" "trl==0.12.2" "transformers<4.48"

# (Optional) confirm we actually have a GPU:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to a T4 runtime")

## 2. Load the base model in 4-bit

We load **`unsloth/Llama-3.2-1B-Instruct`** — Unsloth's recommended free-T4 starter. `load_in_4bit=True` quantizes the weights to 4-bit so the whole thing fits comfortably in the T4's 16 GB.

> **Swapping the base model is a one-line change.** Replace the model name with:
> - `unsloth/gemma-4-E4B-it` — newest small Gemma family, a strong step up in quality, still T4-friendly.
> - `unsloth/gemma-3-270m` — the lightest option, fastest to train, great for a quick classroom demo.


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    # dtype=None lets Unsloth auto-pick (float16 on T4)
    dtype=None,
)


## 3. Attach LoRA adapters (this is the QLoRA part)

**What is LoRA / QLoRA?** Instead of updating all ~1 billion weights (expensive, memory-hungry, easy to break), we:
- **freeze** the base model,
- inject tiny trainable **adapter** matrices into a few key layers, and train only those (a fraction of a percent of the parameters),
- keep the frozen base in **4-bit** (the *Q* in QLoRA) to save memory.

The result: we get the behavior change we want (the Kestrel voice) while training only a few million parameters — fast, cheap, and it fits on a free T4.

The settings below (`r=16`, `lora_alpha=16`, the seven `target_modules`) are Unsloth's own recommended defaults.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # long-context friendly, saves memory
    random_state=3407,
)


## 4. The data is 80% of the job — and its *shape* decides learned vs. instructed

Fine-tuning quality lives and dies on the dataset. Ours is **`kestrel_support_dataset.jsonl`** — ~190 chat-format examples where **every assistant reply** follows the identical five-part Kestrel format. Two deliberate design choices make this a real *learned-format* demo rather than a parlor trick:

1. **The format appears ONLY in the assistant replies — never in the system or user turns.** If we told the model the format in the system prompt, it could pass just by *following instructions*, and the behavior would evaporate the moment you drop that instruction at inference. By keeping the format out of the prompt, the only way for the model to lower its loss is to **internalize** the format. That is what "fine-tuning taught the voice" actually means.
2. **The system prompt varies — including examples with no system prompt at all.** We rotate a few neutral identity prompts ("You are Kestrel Home Support…") and leave ~25% of examples with no system message. This teaches the format as intrinsic to *being* Kestrel Home Support, robust to however it's addressed. Principle: **train the way you'll infer** — if you'll query it with a bare prompt, train it on bare prompts.

The set is built from a hand-curated core (`kestrel_support_dataset.curated.jsonl`, 118 natural Q→reply pairs) plus programmatic augmentation from the canonical policy facts — see **`build_dataset.py`**, which stamps every reply through one template so the shape is byte-for-byte consistent, and `check_format.py`, which scores adherence (the set is 100%).

Upload `kestrel_support_dataset.jsonl` to the Colab session (drag it into the file panel on the left), then run the cell below. We:
1. load it with `datasets.load_dataset("json", ...)`,
2. render each example with the tokenizer's **chat template** (`tokenizer.apply_chat_template`) — which handles examples with or without a system turn,
3. print one formatted example so you can see what the model actually trains on.


In [ ]:
from datasets import load_dataset

# If you're running in Colab, upload the file first (file panel on the left),
# or fetch it from wherever you host the repo.
dataset = load_dataset("json", data_files="kestrel_support_dataset.jsonl", split="train")
print("examples:", len(dataset))

def format_chat(example):
    # Render the system/user/assistant messages into a single training string
    # using the model's own chat template.
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)

# Show ONE fully-formatted example — this is literally what the model sees.
print("\n===== one formatted training example =====\n")
print(dataset[0]["text"])


## 5. Train with TRL's SFTTrainer

We use `SFTTrainer` (supervised fine-tuning) with Unsloth's recommended demo hyperparameters. `max_steps=60` is intentionally short — enough for a 1B model to lock onto a consistent format in a few minutes on a T4.

> **Watch the training loss.** A healthy run settles around **0.5–1.0**. If it plunges toward **0**, the model is memorizing/overfitting — reduce steps or add data variety. For teaching the *format*, a modest loss that stays in the healthy band is exactly what we want.

> **Why `SFTConfig` and not `transformers.TrainingArguments`?** Newer TRL wants a `SFTConfig` (it subclasses `TrainingArguments`, so the fields are the same). Passing a plain `TrainingArguments` makes TRL convert it internally, and combined with Unsloth's module patching that can leave two copies of the `SFTConfig` class around — which crashes with a `PicklingError` when the trainer saves the checkpoint at the end. Passing `SFTConfig` directly avoids it. See the README's Troubleshooting section if you still hit it.


In [ ]:
# NOTE: use TRL's SFTConfig (not transformers.TrainingArguments) as `args`.
# Newer TRL + Unsloth's module patching can otherwise end up with two copies of
# the SFTConfig class, which blows up when the trainer pickles its config at
# checkpoint-save time:
#   PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>:
#   it's not the same object as trl.trainer.sft_config.SFTConfig
# SFTConfig subclasses TrainingArguments, so every field below is identical.
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # ~190 examples at total batch size 8 -> ~24 steps/epoch.
        # 90 steps ~= 3.8 epochs — enough to internalize the format now that the
        # prompt no longer hints it, without memorizing. Watch the loss: healthy
        # is 0.5-1.0; if it dives toward 0, cut steps back.
        max_steps=90,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. BEFORE vs AFTER

Here's the payoff. We ask the **fine-tuned** model a normal Kestrel question and expect it to answer in the exact five-part house format.

**What the untuned base model would produce (illustrative — the base weights are now adapted, so we describe it rather than run it):**

> *A generic, free-form paragraph.* Something like: "The restocking fee for returns after 60 days is typically a percentage of the purchase price. Please check your retailer's return policy..." — no greeting line, no `Answer:` / `Policy:` / `Next step:` structure, no `— Kestrel Home Support` sign-off. It doesn't know Kestrel's voice.

**What the fine-tuned model produces (run the cell below):** the Kestrel format — greeting, `Answer:`, `Policy:`, `Next step:`, and the `— Kestrel Home Support` sign-off.

> To capture a true side-by-side live, run this generation cell **before** training on a second copy of the base model in a separate notebook. In this single-notebook demo we describe the before and *run* the after, because after `get_peft_model` + `train()` the in-memory model is already the tuned one.


In [ ]:
FastLanguageModel.for_inference(model)  # 2x faster generation

# IMPORTANT: this system prompt does NOT describe the Kestrel format — and the
# training data never put the format in the prompt either. The five-part format
# lives ONLY in the assistant replies the model was trained on. So if the tuned
# model still emits greeting / Answer: / Policy: / Next step: / sign-off here,
# that is a LEARNED behavior, not instruction-following. (Run the same prompt on
# the untuned base model and you'll get generic, formatless prose.)
#
# Want to prove it even harder? Set SYSTEM = "" (no system prompt at all) — the
# training set includes plenty of no-system-prompt examples, so the format
# should still hold.
SYSTEM = "You are Kestrel Home Support, the assistant for Kestrel Home smart thermostats."

def kestrel_reply(question, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text.strip()

print(kestrel_reply("What's the restocking fee on a return after 60 days?"))

## 7. The teaching beat: voice, not facts

Now we ask the fine-tuned model about a **fact it never saw in training** — a made-up new product. Watch what happens: it will confidently reply in the perfect Kestrel format... and get the *fact* wrong (it will invent a price, because that number was never in the training data).


In [ ]:
# The 'Kestrel Zephyr' does not exist and its price was never in the training data.
print(kestrel_reply("How much does the new Kestrel Zephyr thermostat cost?"))


### Punchline

See it? The reply is in flawless Kestrel voice — greeting, `Answer:`, `Policy:`, `Next step:`, sign-off — but the **price is made up**. The model never learned that fact, and fine-tuning can't conjure facts it wasn't taught.

> **Fine-tuning taught the VOICE, not the FACTS. For facts, use RAG (last session).**
>
> **Production = fine-tune for voice + RAG for facts.** The fine-tune makes every answer *sound* like Kestrel; RAG feeds the model the *real, current* Kestrel facts (prices, policies, SLAs) at query time so the content is correct too.


## 8. (Optional) Export to GGUF + run in Ollama

To run your fine-tuned model locally (no GPU needed for inference), export it to **GGUF** and load it in **Ollama**.


In [ ]:
# Export a 4-bit (q4_k_m) GGUF you can run anywhere with Ollama / llama.cpp.
result = model.save_pretrained_gguf("kestrel-support", tokenizer, quantization_method="q4_k_m")

# Unsloth writes the GGUF into a SEPARATE "<name>_gguf/" folder and names the file
# after the BASE model (e.g. llama-3.2-1b-instruct.Q4_K_M.gguf), NOT "unsloth.*".
# It also auto-generates a matching Modelfile. Don't guess the paths — they're in
# the returned dict, so print them:
print("GGUF file :", result["gguf_files"][0])       # -> kestrel-support_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf
print("Modelfile :", result["modelfile_location"])  # -> kestrel-support_gguf/Modelfile

# In Colab the files live under /content/. To pull the .gguf onto your machine:
#   from google.colab import files
#   files.download(result["gguf_files"][0])

# Then, on any machine with Ollama installed, use the Modelfile Unsloth generated:
#   1) ollama create kestrel-support -f kestrel-support_gguf/Modelfile
#   2) ollama run kestrel-support "What's the restocking fee after 60 days?"
#
# (Or write your own Modelfile pointing at the exact .gguf path printed above:
#   FROM ./kestrel-support_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf )

### No-code path: Unsloth Studio

Prefer clicking to coding? **Unsloth Studio** is a local no-code UI for the exact same workflow.

```bash
curl -fsSL https://unsloth.ai/install.sh | sh
unsloth studio
```

Then in the browser UI: **upload `kestrel_support_dataset.jsonl`**, pick the base model, click train, and use **Model Arena** to compare the base model vs your fine-tuned model side-by-side. It's the ideal path for non-coders and for a live classroom demo where you'd rather show a UI than a stack trace.


## 9. Beyond SFT: preference tuning with DPO

SFT (sections 1–8) taught the model the Kestrel *format* by showing it ~120 examples of the "right" reply. But SFT only ever sees good examples — it never learns what a **bad** reply looks like, so it can still drift: ramble, skip the `Policy:` line, invent a fee, or (worst of all) fail to escalate a fire/smoke case.

**DPO (Direct Preference Optimization)** is the step beyond SFT. Instead of single "correct" replies, you feed the model **pairs**:

- `prompt` — the support question
- `chosen` — a reply in the correct five-part format with the *correct* policy/escalation
- `rejected` — a plausible-but-worse reply (off-format, wrong fee, or fails to escalate a safety case)

DPO nudges the weights to make the `chosen` reply *more* likely and the `rejected` reply *less* likely — it teaches the model to **prefer good over bad**, not just to imitate good. Think of SFT as "here's how to talk" and DPO as "here's what to avoid."

Our preference set is **`kestrel_preference_dataset.jsonl`** (~24 pairs) covering warranty, restocking, SLAs, and escalation. For example, one `rejected` reply calmly suggests "reset the device" for a *sparking, burning* unit — exactly the failure DPO should train away.

> **When to reach for DPO:** after SFT has locked in the format and you have specific failure modes you want to suppress. It's the standard "polish" pass in the modern post-training stack (SFT → DPO).

### The DPO pattern (next-step, not required for the basic demo)

The snippet below is **illustrative** — it shows how you'd continue from the *already SFT-tuned* `model` above into a DPO pass using TRL's `DPOTrainer` with Unsloth. You do **not** need to run it for the core "voice" demo; it's the pattern for the next level. TRL's `DPOTrainer` expects a dataset with `prompt` / `chosen` / `rejected` columns — which is exactly the shape of `kestrel_preference_dataset.jsonl`.

In [ ]:
# --- ILLUSTRATIVE: DPO preference tuning (the step after SFT) ---
# Not required for the basic voice demo. Run only after the SFT pass above,
# reusing the same in-memory `model` (now SFT-tuned) and `tokenizer`.
from unsloth import PatchDPOTrainer
PatchDPOTrainer()  # makes TRL's DPOTrainer Unsloth-aware (call before importing it)

from trl import DPOTrainer, DPOConfig
from datasets import load_dataset

# Our preference pairs: prompt / chosen / rejected — exactly what DPOTrainer wants.
pref_dataset = load_dataset(
    "json", data_files="kestrel_preference_dataset.jsonl", split="train"
)
print("preference pairs:", len(pref_dataset))

dpo_trainer = DPOTrainer(
    model=model,               # the SFT-tuned model from section 5
    ref_model=None,            # Unsloth/PEFT reuses the base as the frozen reference
    tokenizer=tokenizer,
    train_dataset=pref_dataset,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,          # short: DPO is a light polish on top of SFT
        learning_rate=5e-6,    # much smaller LR than SFT — DPO nudges, doesn't reshape
        beta=0.1,              # how strongly to prefer `chosen` over `rejected`
        logging_steps=1,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_dpo",
        report_to="none",
    ),
)

# dpo_trainer.train()   # <- uncomment to actually run the preference pass
#
# After DPO, re-run the generation cells from section 6: the replies should
# stay in the five-part format AND avoid the `rejected`-style mistakes
# (e.g. it now escalates a "sparking / burning smell" report instead of
# suggesting a reset).